# Slave

In [11]:
%reset -f

In [12]:
from pynq import PL
from pynq import (allocate, Overlay)
import numpy as np
from PIL import Image

PL.reset()

In [13]:
ol = Overlay('slave-zcu102.bit')

In [14]:
#help(ol)

In [15]:
def config_img2axis(ip,buffer, eos,frame_cnt):
    
# Configure registers:
    # Write physical address of buffer to data_port register
    ip.register_map.data_port = buffer.physical_address

    # Set end_of_stream 
    ip.register_map.end_of_stream = eos

    # Set frame_no to 88
    ip.register_map.frame_cnt = frame_cnt
    
    # Master starts the IP core by setting the ap_start bit in CTRL register
    #ip.register_map.CTRL.AP_START=1
    
def image_to_RGB(image_fname):
    # === LOAD AND CONVERT IMAGE TO RGB ===
    img = Image.open(f"{image_fname}").convert('RGB') 
    img_np = np.array(img)  # Shape: (H, W, 3), dtype=uint8

    # === PACK RGB TO INT32 ===
    # Format: 0x00RRGGBB (most significant byte can be 0)
    r = img_np[:, :, 0].astype(np.uint32)
    g = img_np[:, :, 1].astype(np.uint32)
    b = img_np[:, :, 2].astype(np.uint32)
    rgb_packed = (b << 16) | (g << 8) | r  # Shape: (H, W)

    # Allocate contiguous buffer with dtype uint32
    buffer = allocate(shape=rgb_packed.shape, dtype=np.uint32)

    # Copy packed pixels into buffer
    np.copyto(buffer, rgb_packed)

    print(f"Packed buffer shape: {buffer.shape}, dtype: {buffer.dtype}")
    return buffer

In [16]:
from pynq.lib.video import *
vdma = ol.axi_vdma_0

vdma.readchannel.reset()
vdma.readchannel.mode = VideoMode(width=160, height=480, bits_per_pixel=32)

vdma.readchannel.start()

In [17]:
print(f"VDMA.running={vdma.readchannel.running},\nVDMA.activeframe={vdma.readchannel.activeframe},\nVDMA.mode={vdma.readchannel.mode}")

VDMA.running=True,
VDMA.activeframe=0,
VDMA.mode=VideoMode: width=160 height=480 bpp=32 fps=60


In [18]:
img_fname='test_image.jpg'
buff_o=image_to_RGB(img_fname)

Packed buffer shape: (1080, 1920), dtype: uint32


In [19]:
config_img2axis(ol.img2axis_0,buff_o,True,4)

## that's all folsk